# Calibration sweep analysis — φ_f^th, partial molar volumes, λ(φ_p, P), ΔV_mix(P)

Analyses the `simulations/calibration_sweep/` output (see its README for the full
math and how to run the sweep). **Supersedes `volume_of_mixing.ipynb`** (archived).

Per pressure $P$ the sweep holds one fixed network ($N_p$ const) at a grid of exact
solvent counts $N_f$. This notebook:

1. **Sync** — stages `box_dimensions`, `pressure_tensor`, and `calib_*.lammpstrj`
   from Expanse into `flow_data_local/calibration_sweep/` (skip-if-present).
2. **Per run** — block-averaged $\langle V\rangle$ ± SE and the **aniso gate**
   ($P_{xx}\approx P_{yy}\approx P_{zz}$, stable aspect ratio; off-diagonals flagged only).
3. **Per pressure** — weighted polynomial fit of $\ln\langle V\rangle$ vs $\ln N_f$,
   differentiated analytically with CI propagation:
   $\varphi_f^{th} = \partial\ln\langle V\rangle/\partial\ln N_f|_P$,
   $\bar v_f = \varphi_f^{th} V/N_f$, $\bar v_p = (V - N_f\bar v_f)/N_p$ (Euler closure).
4. **Voronoi** — `lib/volfrac.py` whole-box pass on the calibration dumps → $\varphi_f^{vor}$ ± σ.
5. **Joint fit** — $\lambda(\varphi_p, P) = 1 + (a_1+b_1P)\varphi_p + (a_2+b_2P)\varphi_p^2$
   (exact anchor $\lambda(0,P)=1$) → **`scripts/calibration/calibration_lambda.json`**.
6. **Figures** — λ surface + CIs, $\bar v_f(\varphi_p,P)$ and $\bar v_f/v_f^\circ(P)$,
   ΔV_mix(P) over the full range, Euler-closure check, isotropy summary, z-seam check.

Production notebooks consume the json via `volfrac.load_calibration()` /
`volfrac.phi_calibrated()`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json, sys, os, subprocess, datetime
from pathlib import Path
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

%matplotlib inline
%config InlineBackend.figure_format = 'retina'

plt.rcParams.update({
    'font.family':        'CMU Serif',
    'mathtext.fontset':   'cm',
    'mathtext.rm':        'CMU Serif',
    'font.size':          16,
    'axes.titlesize':     18,
    'axes.labelsize':     18,
    'xtick.labelsize':    15,
    'ytick.labelsize':    15,
    'legend.fontsize':    14,
    'axes.unicode_minus': False,
    'figure.dpi':         120,
})

# lib/volfrac.py — the single source of truth for estimators + calibration
sys.path.insert(0, str(Path('lib').resolve()))
import volfrac

# --- Configuration -------------------------------------------------------
# KEEP IN SYNC with simulations/calibration_sweep/calibration_sweep.sh and the
# sweep_manifest/sweep_config_*.txt of the sweep being analysed.
BASE_SNAPSHOT = "final_config_slab_support_periodic_5beads_tall_rho04_new_1.0_1.0_14000000.data"
ISOLATED_STEM = "isolated_" + BASE_SNAPSHOT[len("final_config_"):-len(".data")]
INTERACTION   = "1.0_1.0"   # mixed boxes (epsSS_epsSP)
PURE_INTER    = "1.0_0.0"   # pure companions
PRESSURES     = ["0.50", "0.75", "1.00", "1.25", "1.50", "1.75", "2.00"]  # driver strings
NF_GRID       = [53400, 48100, 43300, 39000, 35100, 31600, 28400, 25600, 23000, 20800]
NREPS         = 2
N_P           = 104283       # polymer beads — constant across the whole grid
CALIB_STEPS      = 300000
CALIB_STEPS_LOWP = 500000
LOWP_MAX         = 0.75

def nsteps_of(P):
    return CALIB_STEPS_LOWP if float(P) <= LOWP_MAX else CALIB_STEPS

def stems(P, nf, rep):
    '''(mixed, solvent) dataname stems for one grid point.'''
    return (f"{ISOLATED_STEM}_nf{nf}_pstar{P}_rep{rep}",
            f"{ISOLATED_STEM}_nf{nf}_solvent_only_pstar{P}_rep{rep}")

def pol_stem(P):
    return f"{ISOLATED_STEM}_polymer_only_pstar{P}"

DATA_DIR  = Path("../../flow_data_local/calibration_sweep")
CALIB_DIR = Path("calibration"); CALIB_DIR.mkdir(exist_ok=True)
CALIB_JSON = CALIB_DIR / "calibration_lambda.json"

print(f"Grid: {len(PRESSURES)} pressures x {len(NF_GRID)} loadings x {NREPS} reps")
print(f"Data root: {DATA_DIR.resolve()}")

## Sync from Expanse (skip-if-present)

Stages the three per-run artifacts (`box_dimensions_*.dat`, `pressure_tensor_*.dat`,
`calib_*.lammpstrj`) plus the sweep manifests. The trajectory files are ~13 MB per
mixed run (~4 GB for a full sweep) — the SFTP loop skips anything already present
locally, so reruns only fetch what's new.

In [ ]:
# === Sync calibration data from Expanse (only when needed) ===
import paramiko, getpass, stat

EXPANSE_HOST = "login.expanse.sdsc.edu"
EXPANSE_USER = "dpollard"
STAGE_DIR    = "/home/dpollard/Documents/lammps_runs/calibration_sweep/calib_stage"

def expected_local_files(include_traj=True):
    files = []
    for P in PRESSURES:
        p_dir, ns = DATA_DIR / f"p{P}", nsteps_of(P)
        for nf in NF_GRID:
            for rep in range(1, NREPS + 1):
                mstem, sstem = stems(P, nf, rep)
                files.append(p_dir / f"box_dimensions_{mstem}_{INTERACTION}_{ns}.dat")
                files.append(p_dir / f"pressure_tensor_{mstem}_{INTERACTION}_{ns}.dat")
                files.append(p_dir / f"box_dimensions_{sstem}_{PURE_INTER}_{ns}.dat")
                files.append(p_dir / f"pressure_tensor_{sstem}_{PURE_INTER}_{ns}.dat")
                if include_traj:
                    files.append(p_dir / f"calib_{mstem}_{INTERACTION}_{ns}.lammpstrj")
        files.append(p_dir / f"box_dimensions_{pol_stem(P)}_{PURE_INTER}_{ns}.dat")
        files.append(p_dir / f"pressure_tensor_{pol_stem(P)}_{PURE_INTER}_{ns}.dat")
    return files

needed  = expected_local_files()
missing = [f for f in needed if not f.exists()]

if not missing:
    print(f"All {len(needed)} calibration files already present locally — skipping Expanse login.")
else:
    print(f"{len(missing)}/{len(needed)} needed files missing locally — syncing from Expanse.")

    # Single find (following traj_files symlinks into scratch) builds a
    # basename -> newest-path index; then bucket into p${P}/ dirs by pstar tag.
    # (volmix_sweep.ipynb sync pattern, extended to three file patterns.)
    p_list  = " ".join(PRESSURES)
    stage_script = (
        "SWEEP=~/Documents/lammps_runs/calibration_sweep\n"
        "STAGE=${SWEEP}/calib_stage\n"
        "mkdir -p \"$STAGE\"\n"
        "declare -A NEWEST\n"
        "while IFS= read -r line; do\n"
        "  p=${line#* }; b=${p##*/}\n"
        "  [ -z \"${NEWEST[$b]:-}\" ] && NEWEST[$b]=\"$p\"\n"
        "done < <(find -L \"$SWEEP\" \\( -name 'box_dimensions_*.dat' -o -name 'pressure_tensor_*.dat' "
        "-o -name 'calib_*.lammpstrj' \\) -not -path '*/calib_stage/*' -printf '%T@ %p\\n' 2>/dev/null | sort -rn)\n"
        "echo \"indexed ${#NEWEST[@]} candidate files\"\n"
        f"for P in {p_list}; do\n"
        "  mkdir -p \"$STAGE/p${P}\"\n"
        "  n=0\n"
        "  for b in \"${!NEWEST[@]}\"; do\n"
        "    case \"$b\" in\n"
        "      *_pstar${P}_*|*_pstar${P}.*) cp -p \"${NEWEST[$b]}\" \"$STAGE/p${P}/\" 2>/dev/null && n=$((n+1)) ;;\n"
        "    esac\n"
        "  done\n"
        "  echo \"  P=${P}: $n files staged\"\n"
        "done\n"
        "cp -p \"$SWEEP\"/sweep_manifest/sweep_config_*.txt \"$STAGE/\" 2>/dev/null || true\n"
    )

    password = getpass.getpass(f"Expanse password for {EXPANSE_USER}: ")
    totp     = getpass.getpass("TOTP / verification code: ")

    def auth_handler(title, instructions, prompt_list):
        return [password if "password" in pr.strip().lower() else totp
                for pr, echo in prompt_list]

    print("Connecting to Expanse...")
    transport = paramiko.Transport((EXPANSE_HOST, 22))
    transport.connect()
    transport.auth_interactive(EXPANSE_USER, auth_handler)
    ssh = paramiko.SSHClient(); ssh._transport = transport

    print("Step 1 — staging files on Expanse...")
    _, stdout, stderr = ssh.exec_command("bash -s", get_pty=False)
    stdout.channel.sendall(stage_script.encode()); stdout.channel.shutdown_write()
    print(stdout.read().decode())

    print("Step 2 — downloading via SFTP (skips files already present)...")
    sftp = ssh.open_sftp()
    DATA_DIR.mkdir(parents=True, exist_ok=True)

    def sftp_download_dir(sftp, remote_dir, local_dir):
        local_dir = Path(local_dir); local_dir.mkdir(parents=True, exist_ok=True)
        for entry in sftp.listdir_attr(remote_dir):
            rp, lp = f"{remote_dir}/{entry.filename}", local_dir / entry.filename
            if stat.S_ISDIR(entry.st_mode):
                sftp_download_dir(sftp, rp, lp)
            else:
                if lp.exists() and lp.stat().st_mtime >= entry.st_mtime:
                    continue
                sftp.get(rp, str(lp))

    sftp_download_dir(sftp, STAGE_DIR, DATA_DIR)
    sftp.close(); ssh.close()
    print("Sync complete.")

## Per-run ⟨V⟩ + aniso gate

`GATE_...` knobs live in `volfrac.aniso_gate`'s signature. A run failing
normal-stress isotropy or aspect-ratio stability is **excluded from the fits**
(`use=False`) and reported below — this doubles as the generator-artifact
diagnostic. Off-diagonal flags never exclude a run.

In [ ]:
TAIL_FRAC = 0.5   # averaging tail; VERIFY equilibration from the V(t) traces below

def load_run(p_dir, stem, inter, ns):
    '''box_dimensions + pressure_tensor for one run -> dict or None if missing.'''
    fb = p_dir / f"box_dimensions_{stem}_{inter}_{ns}.dat"
    ft = p_dir / f"pressure_tensor_{stem}_{inter}_{ns}.dat"
    if not fb.exists():
        return None
    s, lx, ly, lz = volfrac.read_box_dimensions(fb)
    V, Vse = volfrac.block_average_volume(s, lx, ly, lz, tail_frac=TAIL_FRAC)
    out = {'V': V, 'V_se': Vse, 'n_rows': len(s),
           'steps': s, 'lx': lx, 'ly': ly, 'lz': lz}
    if ft.exists():
        sp, pt = volfrac.read_pressure_tensor(ft)
        out['gate'] = volfrac.aniso_gate(sp, pt, lx, ly, lz, tail_frac=TAIL_FRAC)
    else:
        out['gate'] = {'passed': True, 'flags': ['pressure_tensor file missing — gate not applied'],
                       'stats': {}}
    return out

rows, missing_runs = [], []
for P in PRESSURES:
    p_dir, ns = DATA_DIR / f"p{P}", nsteps_of(P)
    pol = load_run(p_dir, pol_stem(P), PURE_INTER, ns)
    if pol:
        rows.append({'P': float(P), 'kind': 'polymer', 'nf': None, 'rep': None,
                     **{k: pol[k] for k in ('V', 'V_se')},
                     'use': pol['gate']['passed'], 'flags': '; '.join(pol['gate']['flags'])})
    else:
        missing_runs.append(f"p{P} polymer")
    for nf in NF_GRID:
        for rep in range(1, NREPS + 1):
            mstem, sstem = stems(P, nf, rep)
            for kind, stem, inter in (('mixed', mstem, INTERACTION),
                                      ('solvent', sstem, PURE_INTER)):
                r = load_run(p_dir, stem, inter, ns)
                if r is None:
                    missing_runs.append(f"p{P} nf{nf} rep{rep} {kind}")
                    continue
                rows.append({'P': float(P), 'kind': kind, 'nf': nf, 'rep': rep,
                             'V': r['V'], 'V_se': r['V_se'],
                             'use': r['gate']['passed'],
                             'flags': '; '.join(r['gate']['flags'])})

runs = pd.DataFrame(rows)
print(f"{len(runs)} runs loaded, {len(missing_runs)} missing")
if missing_runs[:10]:
    print("  missing (first 10):", missing_runs[:10])

gated = runs[~runs['use']]
print(f"\nANISO GATE: {len(gated)} run(s) EXCLUDED")
if len(gated):
    display(gated[['P', 'kind', 'nf', 'rep', 'flags']])
flagged = runs[runs['use'] & (runs['flags'] != '')]
print(f"{len(flagged)} run(s) flagged (off-diagonals etc.) but kept")
if len(flagged):
    display(flagged[['P', 'kind', 'nf', 'rep', 'flags']].head(15))

### Equilibration check — ⟨V⟩ traces

Verify the averaging tail is flat (the run-length budget was an assumption;
this is where it gets checked). Worst offenders shown per pressure.

In [ ]:
fig, axes = plt.subplots(1, len(PRESSURES), figsize=(3.2*len(PRESSURES), 3.2),
                         sharey=False, constrained_layout=True)
for ax, P in zip(np.atleast_1d(axes), PRESSURES):
    p_dir, ns = DATA_DIR / f"p{P}", nsteps_of(P)
    for nf in (NF_GRID[0], NF_GRID[len(NF_GRID)//2], NF_GRID[-1]):
        mstem, _ = stems(P, nf, 1)
        fb = p_dir / f"box_dimensions_{mstem}_{INTERACTION}_{ns}.dat"
        if not fb.exists():
            continue
        s, lx, ly, lz = volfrac.read_box_dimensions(fb)
        ax.plot(s/1e3, lx*ly*lz, lw=1, label=f"nf{nf}")
    ax.axvspan(ns*(1-TAIL_FRAC)/1e3, ns/1e3, alpha=0.12, color='k')
    ax.set_title(f"P={P}"); ax.set_xlabel("step / 1000")
np.atleast_1d(axes)[0].set_ylabel(r"$V$ [$\sigma^3$]")
np.atleast_1d(axes)[0].legend(fontsize=9)
plt.show()

## Per-pressure fit: $\ln\langle V\rangle$ vs $\ln N_f$ → $\varphi_f^{th}$, $\bar v_f$, $\bar v_p$

Weighted cubic polynomial (`POLY_DEG=3`) in $x=\ln N_f$, differentiated analytically.
CIs from the fit covariance (delta method on $\partial y/\partial x$). Euler closure
gives $\bar v_p$ — never fit separately. $v_f^\circ(P)$ comes from the pure-solvent
companions ($V_{sol}/N_f$, loading-independent by construction).

In [ ]:
POLY_DEG = 3

def fit_pressure(P):
    '''-> dict with per-loading phi_th, vbar_f, vbar_p (+CIs) at pressure P.'''
    sub = runs[(runs.P == float(P)) & (runs.kind == 'mixed') & runs.use]
    g = sub.groupby('nf').apply(
        lambda d: pd.Series({'V': np.average(d.V, weights=1/d.V_se.clip(lower=1e-9)**2),
                             'V_se': (d.V.std(ddof=1)/np.sqrt(len(d)) if len(d) > 1
                                      else d.V_se.iloc[0])}),
        include_groups=False).reset_index()
    if len(g) < POLY_DEG + 1:
        return None
    x, y = np.log(g.nf.values.astype(float)), np.log(g.V.values)
    sy   = (g.V_se / g.V).values.clip(1e-9)          # d(lnV) = dV/V
    coef, cov = np.polyfit(x, y, POLY_DEG, w=1/sy, cov=True)
    dcoef = np.polyder(coef)
    phi_th = np.polyval(dcoef, x)
    # delta method: derivative wrt each coef of dy/dx at x
    n = POLY_DEG
    Gd = np.array([[ (n-k) * xx**(n-k-1) if n-k-1 >= 0 else 0.0 for k in range(n+1)]
                   for xx in x])
    phi_se = np.sqrt(np.einsum('ij,jk,ik->i', Gd, cov, Gd))
    V, nf = g.V.values, g.nf.values.astype(float)
    vbar_f  = phi_th * V / nf
    vbar_fe = phi_se * V / nf
    vbar_p  = (V - nf * vbar_f) / N_P
    vbar_pe = nf * vbar_fe / N_P
    # v_f0: pure solvent companions
    sol = runs[(runs.P == float(P)) & (runs.kind == 'solvent') & runs.use]
    vf0 = (sol.V / sol.nf).mean() if len(sol) else np.nan
    vf0_se = (sol.V / sol.nf).std(ddof=1) / np.sqrt(len(sol)) if len(sol) > 1 else np.nan
    return {'P': float(P), 'nf': nf, 'V': V, 'V_se': g.V_se.values,
            'phi_th': phi_th, 'phi_se': phi_se,
            'vbar_f': vbar_f, 'vbar_f_se': vbar_fe,
            'vbar_p': vbar_p, 'vbar_p_se': vbar_pe,
            'vf0': vf0, 'vf0_se': vf0_se,
            'coef': coef, 'cov': cov, 'x': x, 'y': y, 'sy': sy}

fits = {P: fit_pressure(P) for P in PRESSURES}
fits = {P: f for P, f in fits.items() if f is not None}
for P, f in fits.items():
    print(f"P={P}: phi_f^th {f['phi_th'].min():.3f}–{f['phi_th'].max():.3f}  "
          f"vbar_f {f['vbar_f'].min():.3f}–{f['vbar_f'].max():.3f}  "
          f"vbar_p {f['vbar_p'].min():.3f}–{f['vbar_p'].max():.3f}  vf0 {f['vf0']:.3f}")
assert all((f['vbar_p'] > 0).all() for f in fits.values()), \
    'Euler closure violated: vbar_p <= 0 somewhere — inspect before proceeding'


## Voronoi pass on the calibration dumps

Whole-box `volfrac.phi_voronoi_traj` on every mixed calib dump → φ_f^vor ± σ per
(P, N_f, rep). Cached to `voronoi_cache.csv` (keyed by file name) — delete a row
or the file to force recompute.

In [ ]:
VOR_CACHE = DATA_DIR / "voronoi_cache.csv"
cache = (pd.read_csv(VOR_CACHE) if VOR_CACHE.exists()
         else pd.DataFrame(columns=['file', 'phi_vor', 'phi_vor_se', 'n_frames']))

new_rows = []
for P in PRESSURES:
    p_dir, ns = DATA_DIR / f"p{P}", nsteps_of(P)
    for nf in NF_GRID:
        for rep in range(1, NREPS + 1):
            mstem, _ = stems(P, nf, rep)
            fn = f"calib_{mstem}_{INTERACTION}_{ns}.lammpstrj"
            if fn in set(cache['file']):
                continue
            fp = p_dir / fn
            if not fp.exists():
                continue
            ts, phis = volfrac.phi_voronoi_traj(fp, verbose=False)
            new_rows.append({'file': fn, 'phi_vor': phis.mean(),
                             'phi_vor_se': phis.std(ddof=1)/np.sqrt(len(phis)) if len(phis) > 1 else np.nan,
                             'n_frames': len(phis)})
            print(f"  {fn}: phi_f^vor = {phis.mean():.4f} ± {new_rows[-1]['phi_vor_se']:.4f} ({len(phis)} frames)")

if new_rows:
    cache = pd.concat([cache, pd.DataFrame(new_rows)], ignore_index=True)
    cache.to_csv(VOR_CACHE, index=False)
print(f"Voronoi results: {len(cache)} runs cached")

def vor_of(P, nf, rep):
    ns = nsteps_of(P)
    mstem, _ = stems(P, nf, rep)
    m = cache[cache.file == f"calib_{mstem}_{INTERACTION}_{ns}.lammpstrj"]
    return (m.phi_vor.iloc[0], m.phi_vor_se.iloc[0]) if len(m) else (np.nan, np.nan)

### Validation 7 — z-seam homogeneity

The isolated gel's network bonds wrap x,y but **not** z (WCA contact with its own
image at the z boundary). A solvent-rich layer at the seam (likelier at low P)
invalidates that box. Solvent/polymer z-density from the calib frames, seam band
(±2σ around the z faces) vs interior.

In [ ]:
SEAM_BAND  = 2.0    # sigma from each z face
SEAM_EXCESS = 0.15  # flag if seam solvent density exceeds interior by >15%

seam_rows = []
for P in PRESSURES:
    p_dir, ns = DATA_DIR / f"p{P}", nsteps_of(P)
    for nf in NF_GRID:
        mstem, _ = stems(P, nf, 1)   # rep1 is representative — same input file
        fp = p_dir / f"calib_{mstem}_{INTERACTION}_{ns}.lammpstrj"
        if not fp.exists():
            continue
        frames = volfrac.stream_traj_frames(fp)
        ratios = []
        for t, (box, typ, xyz) in frames.items():
            zlo, zhi = box['z']; Lz = zhi - zlo
            z = zlo + (xyz[:, 2] - zlo) % Lz
            sol = z[typ == 3]
            seam = ((z[typ == 3] < zlo + SEAM_BAND) | (z[typ == 3] > zhi - SEAM_BAND)).mean()
            frac_band = 2 * SEAM_BAND / Lz
            ratios.append(seam / frac_band)   # 1.0 = homogeneous
        r = np.mean(ratios)
        seam_rows.append({'P': float(P), 'nf': nf, 'seam_ratio': r,
                          'flag': r > 1 + SEAM_EXCESS})
seam = pd.DataFrame(seam_rows)
if len(seam):
    bad = seam[seam.flag]
    print(f"z-seam check: {len(bad)}/{len(seam)} boxes flagged (seam solvent excess > {SEAM_EXCESS:.0%})")
    if len(bad):
        display(bad)
        print("Flagged boxes are INVALID — fall back to z-wrapping bonds "
              "(bulk variant of slab_with_support_periodic generator) per the plan addendum.")
else:
    print("no calib trajectories found yet")

## Joint weighted fit — $\lambda(\varphi_p, P)$

Data: $\lambda = \varphi_f^{th}/\varphi_f^{vor}$ at each (P, N_f, rep), regressed on
$\varphi_p^{vor} = 1-\varphi_f^{vor}$ (the variable production applies it to).
Model $\lambda - 1 = (a_1+b_1P)\varphi_p + (a_2+b_2P)\varphi_p^2$ — the anchor
$\lambda(0,P)=1$ is structural (no intercept). Promote the P-order only if
residuals demand it.

In [ ]:
tbl = []
for P in PRESSURES:
    f = fits.get(P)
    if f is None:
        continue
    for i, nf in enumerate(f['nf'].astype(int)):
        for rep in range(1, NREPS + 1):
            pv, pv_se = vor_of(P, nf, rep)
            if not np.isfinite(pv):
                continue
            lam   = f['phi_th'][i] / pv
            # error propagation: (d lam)^2 = (dphi_th/pv)^2 + (phi_th dpv / pv^2)^2
            lam_se = np.hypot(f['phi_se'][i] / pv, f['phi_th'][i] * pv_se / pv**2)
            tbl.append({'P': float(P), 'nf': int(nf), 'rep': rep,
                        'phi_th': f['phi_th'][i], 'phi_th_se': f['phi_se'][i],
                        'phi_vor': pv, 'phi_vor_se': pv_se,
                        'phi_p_vor': 1 - pv, 'lam': lam, 'lam_se': lam_se,
                        'V': f['V'][i], 'V_se': f['V_se'][i],
                        'vbar_f': f['vbar_f'][i], 'vbar_p': f['vbar_p'][i],
                        'vf0': f['vf0']})
lamdf = pd.DataFrame(tbl)
print(f"{len(lamdf)} (P, N_f, rep) calibration points")
print(f"|lambda - 1| range: {np.abs(lamdf.lam-1).min():.3f} – {np.abs(lamdf.lam-1).max():.3f}")
if (np.abs(lamdf.lam - 1) > 0.2).any():
    print("WARNING: tens-of-percent corrections — STOP and investigate before trusting "
          "the surface (radical tessellation only then; see plan validation 3).")

# weighted least squares, no intercept
X = np.column_stack([lamdf.phi_p_vor, lamdf.P*lamdf.phi_p_vor,
                     lamdf.phi_p_vor**2, lamdf.P*lamdf.phi_p_vor**2])
yv = (lamdf.lam - 1).values
w  = 1 / lamdf.lam_se.clip(lower=1e-6).values**2
W  = np.diag(w)
XtWX = X.T @ W @ X
beta = np.linalg.solve(XtWX, X.T @ W @ yv)
resid = yv - X @ beta
dof   = len(yv) - 4
s2    = (resid**2 * w).sum() / dof
cov_b = np.linalg.inv(XtWX) * s2
a1, b1, a2, b2 = beta
print(f"\nlambda(phi_p, P) = 1 + ({a1:+.4f} {b1:+.4f}·P)·phi_p + ({a2:+.4f} {b2:+.4f}·P)·phi_p²")
print(f"reduced chi² = {s2:.2f}   (≫1 ⇒ model too stiff — consider promoting the P-order)")

### Save `calibration/calibration_lambda.json`

Coefficients + covariance + the full raw table + metadata. This file is the ONLY
artifact production consumes (`volfrac.load_calibration()`).

In [ ]:
def _git_rev():
    try:
        return subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'],
                                       cwd='..', text=True).strip()
    except Exception:
        return 'unknown'

artifact = {
    'model': 'lambda(phi_p, P) = 1 + (a1 + b1*P)*phi_p + (a2 + b2*P)*phi_p^2',
    'coeffs': {'a1': float(a1), 'b1': float(b1), 'a2': float(a2), 'b2': float(b2)},
    'covariance': {'order': ['a1', 'b1', 'a2', 'b2'], 'matrix': cov_b.tolist()},
    'fit': {'n_points': int(len(lamdf)), 'reduced_chi2': float(s2), 'poly_deg_lnV': POLY_DEG},
    'raw_table': lamdf.to_dict(orient='list'),
    'metadata': {
        'date': datetime.datetime.now().isoformat(timespec='seconds'),
        'interaction': INTERACTION, 'pure_interaction': PURE_INTER,
        'network_file': BASE_SNAPSHOT, 'N_p': N_P,
        'pressures': [float(p) for p in PRESSURES], 'nf_grid': NF_GRID, 'nreps': NREPS,
        'git_rev': _git_rev(),
        'gate_excluded_runs': int((~runs['use']).sum()),
    },
}
with open(CALIB_JSON, 'w') as fjs:
    json.dump(artifact, fjs, indent=2)
print(f"Wrote {CALIB_JSON.resolve()}")

# round-trip through the production loader
calib = volfrac.load_calibration(CALIB_JSON)
assert np.isclose(volfrac.lambda_of(0.0, 1.0, calib), 1.0)
print("volfrac.load_calibration round-trip OK; anchor lambda(0,P)=1 exact")

## Figures

λ surface + CIs · v̄_f(φ_p, P) and v̄_f/v_f°(P) ("inside vs outside") ·
ΔV_mix(P) over the full range (successor to volume_of_mixing.ipynb's plot) ·
Euler closure · isotropy summary.

In [ ]:
cmap = plt.cm.viridis
pmin, pmax = float(PRESSURES[0]), float(PRESSURES[-1])
pcol = lambda P: cmap((float(P) - pmin) / max(pmax - pmin, 1e-9))

fig, axes = plt.subplots(2, 3, figsize=(19, 10), constrained_layout=True)

# (a) lambda surface
ax = axes[0, 0]
pp = np.linspace(0, lamdf.phi_p_vor.max()*1.05, 100)
for P in PRESSURES:
    Pf = float(P)
    sub = lamdf[lamdf.P == Pf]
    ax.errorbar(sub.phi_p_vor, sub.lam, yerr=sub.lam_se, fmt='o', ms=4,
                color=pcol(P), alpha=0.8)
    Xp = np.column_stack([pp, Pf*pp, pp**2, Pf*pp**2])
    lam_fit = 1 + Xp @ beta
    se_fit  = np.sqrt(np.einsum('ij,jk,ik->i', Xp, cov_b, Xp))
    ax.plot(pp, lam_fit, color=pcol(P), lw=1.5, label=f"P={P}")
    ax.fill_between(pp, lam_fit-se_fit, lam_fit+se_fit, color=pcol(P), alpha=0.15)
ax.axhline(1, color='k', lw=0.8, ls=':')
ax.set_xlabel(r"$\varphi_p^{vor}$"); ax.set_ylabel(r"$\lambda$")
ax.set_title(r"calibration surface $\lambda(\varphi_p, P)$"); ax.legend(fontsize=9, ncol=2)

# (b) vbar_f(phi_p, P)
ax = axes[0, 1]
for P, f in fits.items():
    phi_p_th = 1 - f['phi_th']
    ax.errorbar(phi_p_th, f['vbar_f'], yerr=f['vbar_f_se'], fmt='o-', ms=4,
                color=pcol(P), label=f"P={P}")
ax.set_xlabel(r"$\varphi_p^{th}$"); ax.set_ylabel(r"$\bar v_f$ [$\sigma^3$]")
ax.set_title(r"solvent partial molecular volume"); ax.legend(fontsize=9, ncol=2)

# (c) vbar_f / vf0 (inside vs outside)
ax = axes[0, 2]
for P, f in fits.items():
    ax.errorbar(1 - f['phi_th'], f['vbar_f'] / f['vf0'], yerr=f['vbar_f_se']/f['vf0'],
                fmt='o-', ms=4, color=pcol(P), label=f"P={P}")
ax.axhline(1, color='k', lw=0.8, ls=':')
ax.set_xlabel(r"$\varphi_p^{th}$"); ax.set_ylabel(r"$\bar v_f / v_f^\circ$")
ax.set_title("inside vs outside the gel")

# (d) dV_mix(P) per loading — scale = 1 convention
ax = axes[1, 0]
dv_rows = []
for P in PRESSURES:
    Pf = float(P)
    pol = runs[(runs.P == Pf) & (runs.kind == 'polymer') & runs.use]
    if not len(pol):
        continue
    V_pol = pol.V.iloc[0]
    for nf in NF_GRID:
        m = runs[(runs.P == Pf) & (runs.kind == 'mixed')   & (runs.nf == nf) & runs.use]
        s = runs[(runs.P == Pf) & (runs.kind == 'solvent') & (runs.nf == nf) & runs.use]
        if not (len(m) and len(s)):
            continue
        dv  = m.V.mean() - s.V.mean() - V_pol
        den = s.V.mean() + V_pol
        se  = np.hypot(m.V.sem() if len(m) > 1 else m.V_se.iloc[0],
                       s.V.sem() if len(s) > 1 else s.V_se.iloc[0]) / den
        dv_rows.append({'P': Pf, 'nf': nf, 'ratio': dv/den, 'se': se})
dvdf = pd.DataFrame(dv_rows)
for nf in NF_GRID:
    sub = dvdf[dvdf.nf == nf]
    if len(sub):
        ax.errorbar(sub.P, sub.ratio, yerr=sub.se, fmt='o-', ms=4,
                    color=plt.cm.plasma(1 - (nf - min(NF_GRID))/(max(NF_GRID)-min(NF_GRID))),
                    label=f"nf{nf}", alpha=0.85)
ax.axhline(0, color='k', lw=0.8, ls=':')
ax.set_xlabel(r"$P^*$"); ax.set_ylabel(r"$\Delta V_{mix}/(V_{sol}+V_{pol})$")
ax.set_title(r"volume of mixing (scale = 1)"); ax.legend(fontsize=7, ncol=2)

# (e) Euler closure: vbar_p(phi_p, P) — must be > 0 and smooth
ax = axes[1, 1]
for P, f in fits.items():
    ax.errorbar(1 - f['phi_th'], f['vbar_p'], yerr=f['vbar_p_se'], fmt='o-', ms=4,
                color=pcol(P), label=f"P={P}")
ax.axhline(0, color='r', lw=0.8, ls='--')
ax.set_xlabel(r"$\varphi_p^{th}$"); ax.set_ylabel(r"$\bar v_p$ [$\sigma^3$]")
ax.set_title("Euler-closure check")

# (f) isotropy summary: worst normal-stress spread per (P, kind)
ax = axes[1, 2]
iso_rows = []
for P in PRESSURES:
    Pf = float(P)
    sub = runs[(runs.P == Pf)]
    n_ok, n_flag, n_fail = len(sub[sub.use & (sub.flags == '')]), \
                           len(sub[sub.use & (sub.flags != '')]), len(sub[~sub.use])
    iso_rows.append({'P': Pf, 'ok': n_ok, 'flagged': n_flag, 'failed': n_fail})
iso = pd.DataFrame(iso_rows)
ax.bar(iso.P, iso.ok, width=0.18, label='clean', color='#4daf7c')
ax.bar(iso.P, iso.flagged, width=0.18, bottom=iso.ok, label='flagged', color='#e8b23a')
ax.bar(iso.P, iso.failed, width=0.18, bottom=iso.ok+iso.flagged, label='gate-failed', color='#d64545')
ax.set_xlabel(r"$P^*$"); ax.set_ylabel("runs"); ax.set_title("aniso-gate summary"); ax.legend(fontsize=9)

fig.savefig(CALIB_DIR / 'calibration_summary.png', bbox_inches='tight')
plt.show()
print(f"Saved {CALIB_DIR/'calibration_summary.png'}")

## Validation checklist (plan Phase 5 — must pass before production use)

| # | Check | Where |
|---|---|---|
| 1 | Reservoir bins: φ_f^cal ≡ φ_f^vor ≡ 1 within noise | production notebooks after wiring (anchor is structural here) |
| 2 | Self-consistency: slab interior vs independent φ_f^th at same state, within CI | production §1b once wired |
| 3 | λ surface smooth; \|λ−1\| ≲ few % (tens of % ⇒ STOP, investigate; radical tessellation only then) | cell above (warning printed) |
| 4 | Every calibration run passed the aniso gate | gate summary above |
| 5 | ΔV_mix(P) over full [0.5, 2.0] as its own figure/table | panel (d) + `lamdf`/`dvdf` tables |
| 6 | Interface bins (large ∇φ_p^vor) flagged; report raw + calibrated | production notebooks |
| 7 | z-seam homogeneity (bonds don't wrap z) | seam cell above; fallback = z-wrapping bonds variant |
